## Business Problem

A SaaS subscription company is experiencing customer churn.

Goal:
- Identify high-risk customer segments
- Quantify revenue impact
- Recommend actions to reduce churn in next 90 days

## Assumptions & Limitations

- The dataset provides only a static cross-sectional snapshot with no exact signup or churn event timestamps, making time-series analysis, cohort tracking, or true survival modeling impossible.
- Marketing spend, customer acquisition cost (CAC), and campaign attribution data are absent, preventing any LTV:CAC analysis or evaluation of acquisition channel effectiveness.
- The churn rate calculation carries **length bias**: longer-tenured customers are over-represented in the retained group, while newer customers have had less exposure time to churn, potentially understating true churn risk for recent sign-ups.
- The overall average MRR blends active and churned customers, which overstates current revenue health. Once churned, a customer contributes $0 MRR; including their historical MRR inflates the metric and gives a misleading view of the active revenue pipeline.

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [7]:
df = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Initial Observations
- This data has zero null values.
- `TotalCharges` has object data type which needs conversion.

In [16]:
print(f"DataSet Shape is : {df.shape} \n")
print("Data quick info:")
df.info()
print("\nNull Count:")
df.isnull().sum()

DataSet Shape is : (7043, 21) 

Data quick info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

- **Observation:** TotalCharges is stored as an object. It must be coerced to a numeric type to calculate revenue metrics, which will expose any hidden empty strings as nulls.

In [17]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print(f"Nulls after conversion: {df['TotalCharges'].isnull().sum()}")

Nulls after conversion: 11


In [20]:
df[df['TotalCharges'].isnull()]

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.55,NaN,No
753,3115-CZMZD,Male,0,No,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.25,NaN,No
936,5709-LVOEQ,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,...,Yes,No,Yes,Yes,Two year,No,Mailed check,80.85,NaN,No
1082,4367-NUYAO,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.75,NaN,No
1340,1371-DWPAZ,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,...,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.05,NaN,No
3331,7644-OMVMY,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.85,NaN,No
3826,3213-VVOLG,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,NaN,No
4380,2520-SGTTA,Female,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.00,NaN,No
5218,2923-ARZLG,Male,0,Yes,Yes,0,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.70,NaN,No
6670,4075-WKNIU,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Mailed check,73.35,NaN,No


- **Action**: Coerced TotalCharges to numeric. Found 11 nulls. Because these customers have a tenure of 0 (brand new signups), their total charges were filled with 0 rather than dropped, preserving new-user acquisition data.
- **Assumption**: Filled null TotalCharges with 0 assuming postpaid billing (no bill generated yet). Assumption breaks if prepaid (TotalCharges should equal MonthlyCharges).

In [23]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

## Data Summary & Validation

The `.describe()` output confirms a reasonable spread across all numeric columns with no obvious anomalies. MRR ranges from \\$18.25 to \\$118.75, LTV from \\$0 to \\$8,684.80, and MonthsSubscribed from 0 to 72 months, all within expected bounds for a telco/SaaS dataset.

In [48]:
df.describe()

,SeniorCitizen,MonthsSubscribed,MRR,LTV
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


## SaaS Schema Mapping
**Action**: Mapped telco operational columns to standard SaaS business metrics to align analysis with executive terminology.

In [28]:
# renaming columns
rename_dict = {
    "MonthlyCharges":"MRR",
    "TotalCharges":"HistoricalRevenue",
    "tenure":"MonthsSubscribed"
}

df.rename(columns=rename_dict, inplace=True)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,MonthsSubscribed,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MRR,HistoricalRevenue,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [29]:
# Fix the column name
df = df.rename(columns={"HistoricalRevenue": "LTV"})
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,MonthsSubscribed,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MRR,LTV,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Business KPIs: 
- What is the current churn rate?
- overall Average MRR?
- Average MRR segmented by churn status?
- the average lifespan of a churned customer?

In [46]:
# current churn rate %
churn_rate = round((df['Churn'] == "Yes").mean() * 100, 2)

# Overall Average MRR
avg_mrr = round(df['MRR'].mean(), 2)

# Current revenue health (only active customers)
avg_mrr_active = round(df[df['Churn'] == 'No']['MRR'].mean(), 2)

# Avg MRR by churn
avg_mrr_by_churn = round(df.groupby('Churn')['MRR'].mean(),2)

# avg lifespan of churned customers
churned_avg_lifespan = round(df[df['Churn'] == 'Yes']['MonthsSubscribed'].mean(), 2)

print(f"Churn Rate: {churn_rate}%")
print(f"Overall Average MRR (incl. churned): ${avg_mrr}")
print(f"Active Customers Average MRR: ${avg_mrr_active}")
print("\nAverage MRR by Churn:")
print(avg_mrr_by_churn)
print(f"\nAverage Lifespan of Churned Customers: {churned_avg_lifespan} months")

Churn Rate: 26.54%
Overall Average MRR (incl. churned): $64.76
Active Customers Average MRR: $61.27

Average MRR by Churn:
Churn
No     61.27
Yes    74.44
Name: MRR, dtype: float64

Average Lifespan of Churned Customers: 17.98 months


## Key Findings

The overall churn rate stands at 26%. The average MRR across the entire customer base is \\$64.76. Notably, churned customers exhibit a higher average MRR of \\$74.44 compared to non-churned customers at \\$61.27. Additionally, churned customers remained active for an average of 18 months before churning.